In [2]:
import torch
import torch.nn as nn
import torch.optim as optim


In [4]:
# 1. Dataset
X = torch.tensor([
    [2000,480,7000],
    [2500,520,6000],
    [3000,560,5000],
    [3500,610,4000],
    [4200,650,3500],
    [5000,700,2500],
    [6000,750,1500],
    [7500,810,500]
],dtype=torch.float32)

y = torch.tensor([0,0,0,1,1,1,1,1],dtype=torch.float32).view(-1,1)

# 2. Normalizasiya
X_norm = X.clone()
X_norm[:,0]=X[:,0]/10000.0
X_norm[:,1]=X[:,1]/850.0
X_norm[:,2]=X[:,2]/10000.0

# 3. Model
class CreditNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3,8)
        self.fc2 = nn.Linear(8,1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x

model = CreditNet()

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Training loop
epochs = 2000
for epoch in range(epochs):
    y_pred = model(X_norm)
    loss = criterion(y_pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if(epoch+1)%400==0:
       print(f"Epoch {epoch+1}/{epochs}, Loss = {loss.item():.4f}")

# 5. Accuracy hesablayın
with torch.no_grad():
    probs = model(X_norm)
    preds=(probs>0.5).float()
    accuracy = (preds == y).float().mean() * 100
    print(f"\nTrain Accuracy = {accuracy:.2f}%")

# 6. Yeni müştərilər üçün proqnoz verin

new_customers = torch.tensor([
    [2800, 540, 5500],
    [4000, 640, 3000],
    [7000, 790, 1000]
], dtype=torch.float32)

new_norm = new_customers.clone()
new_norm[:,0]=new_norm[:,0]/10000.0
new_norm[:,1]=new_norm[:,1]/850.0
new_norm[:,2]=new_norm[:,2]/10000.0

names = ["A","B","C"]

with torch.no_grad():
    probs = model(new_norm)
    preds=(probs>0.5).float()

    for name,prob,pred in zip(names,probs,preds):
        result = "TESDIQLENDI" if pred.item() == 1 else "TESDIQLENMEDI"
        print(f"Customer={name} | Approval_Prob={prob.item():.4f} | Result={result}")


Epoch 400/2000, Loss = 0.4810
Epoch 800/2000, Loss = 0.2582
Epoch 1200/2000, Loss = 0.1520
Epoch 1600/2000, Loss = 0.1013
Epoch 2000/2000, Loss = 0.0716

Train Accuaracy = 100.00%
Customer=A | Approval_Prob=0.0868 | Result=TESDIQLENMEDI
Customer=B | Approval_Prob=0.9718 | Result=TESDIQLENDI
Customer=C | Approval_Prob=1.0000 | Result=TESDIQLENDI


## Nəzəri suallar

**1. `model = CreditNet()` nə edir?**
Model obyektini yaradır.

**2. `BCELoss` hansı problemlər üçün istifadə olunur?**
Binary Classification (0 və 1) problemləri üçün.

**3. `model.parameters()` optimizer-ə nə verir?**
Modelin öyrənilən parametrlərini (weights və bias).

**4. `lr=0.01` nəyi müəyyən edir?**
Learning rate-i, yəni öyrənmə sürətini.

**5. `optimizer.zero_grad()` niyə hər epoch-da çağırılır?**
Köhnə gradientləri sıfırlamaq üçün.

**6. `loss.backward()` və `optimizer.step()` arasında fərq nədir?**
- `loss.backward()` → Gradientləri hesablayır.
- `optimizer.step()` → Çəkiləri yeniləyir.

**7. Niyə training və yeni məlumatlar eyni qayda ilə normallaşdırılmalıdır?**
Model eyni miqyasda işləməsi üçün.

**8. Bu dataset-in çox kiçik olması hansı problemə səbəb ola bilər?**
Overfitting yarada bilər və yeni məlumatlarda zəif nəticə verə bilər.